# 화질 지표 — 무엇을 재고 있는가

복원 결과가 "좋다" 는 것을 숫자로 말하려면 잣대가 필요하다. 잣대는 두 갈래다.

| | 지표 | 정답 영상이 | 재는 것 |
|---|---|---|---|
| **FR** 참조 있음 | PSNR, SSIM, ERGAS | **있어야 한다** | 정답과 얼마나 다른가 |
| **NR** 참조 없음 | NIQE, BRISQUE, PIQE | **필요 없다** | 이 영상 하나가 얼마나 자연스러운가 |

세 가지를 눈으로 확인한다.

1. 같은 결과에 여섯 지표를 붙여 본다 — **그림 옆에 점수를 나란히 놓는다**
2. **흐리지만 PSNR 높은 결과**와 **선명하지만 PSNR 낮은 결과** — 지각-왜곡 트레이드오프
3. 정답이 없는 인천에서 NR 을 어떻게 읽고 임계값을 어떻게 정할지

미리 뽑아둔 결과를 불러온다. **GPU 가 필요 없다.**

## 1. 준비

In [ ]:
import sys, json, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('sr_utils.py', 'iqa.py'):
    urllib.request.urlretrieve(f'{BASE}/lib/{f}', f)
    sys.modules.pop(f[:-3], None)
from sr_utils import *
import iqa
import pandas as pd

RES = f'{BASE}/results/comparison'
V = 'q2'          # 내려받은 파일 캐시 폴더. 결과가 바뀌면 이 값을 올린다
META = json.load(open(fetch(f'{RES}/meta.json', f'{V}/meta.json')))

# NIQE·BRISQUE 는 "정상 영상이란 이런 것" 이라는 기준이 있어야 점수가 나온다.
# 원 논문은 자연 사진으로 만들지만 위성 영상은 통계가 달라, 이 프로젝트의 HR 로
# 다시 잡은 것을 쓴다. -> 값은 이 데이터 안에서의 상대 비교로만 읽는다.
for f in ('niqe_model.npz', 'brisque_svr.npz'):
    fetch(f'{BASE}/results/iqa/{f}', f'{V}/iqa/{f}')
iqa.load_models(f'{V}/iqa')

VALS = sorted(k for k in META if k.startswith('val'))
TESTS_ = sorted(k for k in META if k.startswith('test'))
ORDER = ['Bicubic', 'SRCNN', 'VDSR', 'EDSR', 'SRGAN', 'ESRGAN', 'SwinIR', 'HAT']
CENTER, SIZE = (67, 370), 90      # 다른 페이지와 같은 확대 자리
CROP = (max(0, CENTER[0] - SIZE // 2), max(0, min(CENTER[1] - SIZE // 2, 384 - SIZE)),
        SIZE)                     # (x, y, 크기) — 384px 패치 안으로 당긴 값


def load(scene, name):
    return imageio.imread(fetch(f'{RES}/{scene}/{name}.png', f'{V}/{scene}/{name}.png'))


print('정답 있는 패치:', ', '.join(VALS))
print('정답 없는 인천:', ', '.join(TESTS_))

## 2. 정답이 있을 때와 없을 때

왼쪽 넷은 정답 HR 이 있어 여섯 지표를 모두 잴 수 있다.
오른쪽 둘은 실제 Sentinel-2 촬영본이라 정답이 없다 — **FR 세 가지는 아예 계산할 수 없다.**

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12, 8.2))
for j, v in enumerate(VALS):
    a = ax[j // 3][j % 3]
    a.imshow(load(v, 'HR')); a.set_title(f'{v}   HR available', fontsize=11)
for j, t in enumerate(TESTS_):
    a = ax[1][j + 1]
    a.imshow(load(t, 'Bicubic')); a.set_title(f'{t}   no HR', fontsize=11, color='#c0504d')
for r in range(2):
    for c in range(3):
        ax[r][c].set_xticks([]); ax[r][c].set_yticks([])
plt.tight_layout(); plt.show()

## 3. 그림 옆에 점수를 붙여 본다

표만 보면 어느 숫자가 어느 그림인지 이어지지 않는다. 확대한 결과 바로 옆에 여섯 점수를
놓는다. **칸 색은 그 지표 안에서의 순위다 — 초록이 좋고 붉을수록 나쁘다.**

화살표가 좋은 방향이다. `↑` 는 높을수록, `↓` 는 낮을수록 좋다.

In [ ]:
v = VALS[0]
hr = load(v, 'HR')
items = [(m, load(v, m), iqa.evaluate(load(v, m), hr)) for m in ORDER]
iqa.report(items, crop=CROP, title=v)

위에서 아래로 훑으면 **초록 칸이 한 줄에 모이지 않는다.** 어떤 모델은 왼쪽 세 칸(FR)이
초록인데 오른쪽 세 칸(NR)이 붉고, 어떤 모델은 정반대다. 지표가 서로 다른 것을 재고 있다는
뜻이다.

검증 4장 전체 평균으로도 확인한다.

In [ ]:
rows = []
for m in ORDER:
    acc = {k: [] for k in iqa.ALL}
    for v in VALS:
        for k, val in iqa.evaluate(load(v, m), load(v, 'HR')).items():
            acc[k].append(val)
    rows.append({'model': m, **{k: float(np.mean(a)) for k, a in acc.items()}})
df = pd.DataFrame(rows).set_index('model')

# 지표마다 1등을 뽑아 본다
print('지표마다 1등이 다르다\n')
for k in iqa.ALL:
    win = df[k].idxmax() if iqa.BETTER[k] == 'high' else df[k].idxmin()
    arrow = '↑' if iqa.BETTER[k] == 'high' else '↓'
    print(f'  {k.upper():8s} {arrow}   1위 {win:8s} ({df.loc[win, k]:.3f})')
print()
print(df.round(3).to_string())

## 4. 지각-왜곡 트레이드오프

3절에서 FR 1등과 NR 1등이 갈렸다. 두 모델만 크게 놓고 본다.

- **FR 이 높은 쪽**: 정답 화소값에 가깝게 가려고 애매한 곳을 평균으로 메운다 → **흐려진다**
- **NR 이 좋은 쪽**: 그럴듯한 질감을 만들어 넣는다 → **선명하지만 정답과 화소가 어긋난다**

In [ ]:
FR_BEST = df['psnr'].idxmax()
NR_BEST = df['niqe'].idxmin()
print(f'PSNR 1위 = {FR_BEST}   (흐린 쪽)')
print(f'NIQE 1위 = {NR_BEST}   (선명한 쪽)\n')

for v in VALS[:2]:
    zoom([('Bicubic', load(v, 'Bicubic')),
          (f'{FR_BEST} (best PSNR)', load(v, FR_BEST)),
          (f'{NR_BEST} (best NIQE)', load(v, NR_BEST)),
          ('Target HR', load(v, 'HR'))],
         center=CENTER, size=SIZE, title=v)

In [ ]:
# 두 모델의 점수를 직접 맞대 본다
two = [(m, load(VALS[0], m), iqa.evaluate(load(VALS[0], m), load(VALS[0], 'HR')))
       for m in (FR_BEST, NR_BEST)]
iqa.report(two, crop=CROP, title=f'{FR_BEST} vs {NR_BEST}', figsize_w=2.4)

In [ ]:
# 가로축 왜곡(PSNR), 세로축 지각(NIQE). 왼쪽 위가 둘 다 좋은 자리인데, 비어 있다
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, yk in zip(ax, ['niqe', 'brisque']):
    a.scatter(df['psnr'], df[yk], s=110, color='#4f7fa8', zorder=3)
    for m, r in df.iterrows():
        a.annotate(m, (r['psnr'], r[yk]), fontsize=10,
                   xytext=(7, 4), textcoords='offset points')
    a.set_xlabel('PSNR  (right = better)', fontsize=11)
    a.set_ylabel(f'{yk.upper()}  (down = better)', fontsize=11)
    a.set_title('nobody is in the top-right corner', fontsize=12)
    a.invert_yaxis(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
print('세로축을 뒤집어 위쪽이 좋게 그렸다. 오른쪽 위가 비어 있다는 것이 트레이드오프다.')

### PIQE 만 반대로 나온다

NIQE·BRISQUE 는 선명한 쪽을 좋게 보는데 **PIQE 는 bicubic 을 1등으로 본다.** 셋 다 NR 인데
방향이 갈린다.

PIQE 가 재는 것은 "이음매와 잡음이 있는가" 다. 흐릿하기만 한 영상은 이음매도 잡음도 없으니
좋은 점수를 받는다. **선명함을 재는 지표가 아니다.**

NR 지표를 쓸 때 가장 먼저 물어야 할 것이 이것이다 — **이 지표는 무엇을 벌점으로 삼는가.**

In [ ]:
rk = pd.DataFrame({k: df[k].rank(ascending=(iqa.BETTER[k] != 'high')).astype(int)
                   for k in ['psnr', 'niqe', 'brisque', 'piqe']})
fig, a = plt.subplots(figsize=(9.0, 4.6))
im = a.imshow(rk.values, cmap='RdYlGn_r', vmin=1, vmax=len(rk))
for i in range(len(rk)):
    for j in range(rk.shape[1]):
        a.text(j, i, rk.values[i, j], ha='center', va='center', fontsize=12)
a.set_xticks(range(rk.shape[1]))
a.set_xticklabels([c.upper() for c in rk.columns], fontsize=11)
a.set_yticks(range(len(rk))); a.set_yticklabels(rk.index)
a.set_title('rank per metric  (1 = best)', fontsize=12)
plt.tight_layout(); plt.show()
print('ESRGAN 줄을 보라. NIQE·BRISQUE 는 1등인데 PSNR·PIQE 는 꼴찌다.')

## 5. 정답이 없는 인천 — 어떻게 읽을 것인가

인천은 실제 촬영본이라 정답이 없다. PSNR·SSIM·ERGAS 는 **계산 자체가 불가능하다.**
남는 것은 NR 세 가지뿐이다.

In [ ]:
t = TESTS_[0]
items = [(m, load(t, m), iqa.evaluate(load(t, m))) for m in ORDER]
iqa.report(items, metrics=iqa.NR, crop=(300, 300, 200), title=f'{t}  (no reference)')

### 임계값을 어떻게 정할 것인가

**절대 기준은 없다.** NIQE 3.5 가 "좋다" 는 뜻이 아니다 — 기준 모델을 무엇으로 잡았느냐에
따라 값이 통째로 움직인다. 실제로 쓰려면 기준선을 스스로 만들어야 한다. 세 가지 방법이 있다.

1. **정답을 아는 구간에서 대응표를 만든다** — 검증셋처럼 HR 이 있는 데이터에서 NR 과 FR 을
   함께 재 두면 "NR 이 이 값이면 PSNR 이 대략 얼마" 라는 환산이 생긴다
2. **입력 자체를 바닥으로 삼는다** — 결과가 입력(bicubic)보다 NR 이 나쁘면 무언가 잘못된
   것이다. 정답 없이도 판정할 수 있다
3. **분포에서 상대 위치로 정한다** — 같은 센서·같은 지역 결과를 여러 장 모아 하위 5% 처럼

아래에서 1번과 2번을 직접 해 본다.

In [ ]:
# 1번 — NR 로 FR 을 가늠할 수 있나
pt = []
for m in ORDER:
    for v in VALS:
        r = iqa.evaluate(load(v, m), load(v, 'HR'))
        pt.append((r['psnr'], r['niqe'], r['piqe']))
P = pd.DataFrame(pt, columns=['psnr', 'niqe', 'piqe'])

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, k in zip(ax, ['niqe', 'piqe']):
    c = np.corrcoef(P[k], P['psnr'])[0, 1]
    a.scatter(P[k], P['psnr'], s=45, alpha=.75,
              color='#4f7fa8' if abs(c) > .4 else '#c0504d')
    a.set_xlabel(f'{k.upper()}   (no reference needed)', fontsize=11)
    a.set_ylabel('PSNR   (needs reference)', fontsize=11)
    a.set_title(f'corr {c:+.3f}   ' + ('usable' if abs(c) > .4 else 'NOT usable'),
                fontsize=12)
    a.grid(alpha=.3)
plt.tight_layout(); plt.show()
print('상관이 뚜렷해야 그 NR 값으로 PSNR 을 대신 가늠할 수 있다.')
print('약하거나 부호가 반대면 그 지표는 이 데이터에서 대리 지표로 못 쓴다.')

In [ ]:
# 2번 — 입력(bicubic) 을 바닥으로 삼는 판정
rows = []
for m in ORDER:
    acc = {k: [] for k in iqa.NR}
    for t in TESTS_:
        for k in iqa.NR:
            acc[k].append(iqa.evaluate(load(t, m))[k])
    rows.append({'model': m, **{k: float(np.mean(a)) for k, a in acc.items()}})
dt = pd.DataFrame(rows).set_index('model')
base = dt.loc['Bicubic']

print('인천 — bicubic 보다 나쁜 항목에 표시\n')
print(f'{"":9s}' + ''.join(f'{k.upper():>12s}' for k in iqa.NR))
print(f'{"Bicubic":9s}' + ''.join(f'{base[k]:12.3f}' for k in iqa.NR) + '   <- 바닥')
for m in ORDER[1:]:
    line = f'{m:9s}'
    for k in iqa.NR:
        v = dt.loc[m, k]
        line += f'{v:11.3f}' + ('!' if v > base[k] else ' ')
    print(line)
print('\n! 는 입력보다 나빠진 항목이다. PIQE 에 몰려 있는데, 위에서 본 이유 때문이다.')

## 6. 정리

**지표마다 1등이 다르다.** 3절 카드에서 초록 칸이 한 줄로 모이지 않았다. FR 세 가지는
서로 비슷하게 움직이지만 NR 은 무엇을 벌점으로 삼느냐에 따라 순위가 갈린다.

**지각-왜곡 트레이드오프는 실재한다.** GAN 계열은 PSNR 이 bicubic 보다도 낮은데
NIQE·BRISQUE 는 압도적으로 좋다. 화소를 맞히는 일과 그럴듯해 보이는 일은 서로 다른
목표이고, 한쪽을 올리면 다른 쪽이 내려간다. 산점도의 오른쪽 위가 비어 있는 것이 그 증거다.

**NR 값에는 절대 기준이 없다.** 기준 모델을 무엇으로 잡았느냐에 따라 통째로 움직이므로
다른 논문의 수치와 직접 견주면 안 된다. 쓰려면 정답을 아는 구간에서 대응표를 만들거나,
입력 자체를 바닥으로 삼거나, 분포에서 상대 위치를 본다.

**그래서 무엇을 볼 것인가.** 지도 제작이나 변화 탐지처럼 화소값이 중요한 일이면 FR 을 본다.
사람이 눈으로 판독하는 일이면 NR 과 실제 확대 그림을 함께 본다. **지표 하나로 결론을 내지
않는 것** 이 이 페이지의 요지다.